In [60]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, round, datediff, when, to_timestamp

spark = SparkSession.builder.appName("olist_gold").getOrCreate()

df_orders = spark.read.parquet('../data/silver/orders')
df_order_items = spark.read.parquet('../data/silver/order_items')
df_order_payments = spark.read.parquet('../data/silver/order_payments')
df_order_reviews = spark.read.parquet('../data/silver/order_reviews')
df_customers = spark.read.parquet('../data/silver/customers')
df_products = spark.read.parquet('../data/silver/products')
df_sellers = spark.read.parquet('../data/silver/sellers')
df_category = spark.read.parquet('../data/silver/category')
df_geolocation = spark.read.parquet('../data/silver/geolocation')

In [61]:
df_items_base = df_order_items \
    .join(df_orders, "order_id") \
    .join(df_products, "product_id") \
    .join(df_sellers, "seller_id") \
    .join(df_category, "product_category_name", "left") \
    .join(df_customers, "customer_id")

In [62]:
df_gold_sales = df_items_base \
    .join(df_customers.select("customer_id", "customer_state").alias("cust"), "customer_id") \
    .select(
        "order_id", "order_item_id", "product_id", "seller_id", "seller_state",
        "price", "freight_value", "order_status",
        "order_purchase_timestamp", "order_delivered_customer_date",
        "order_estimated_delivery_date", "product_category_name_english",
        "customer_id", "cust.customer_state"
    )
df_gold_sales.write.parquet('../data/gold/gold_sales', mode='overwrite')

26/06/18 14:28:00 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


In [63]:
df_gold_delivery_satisfaction = df_items_base \
    .join(df_customers.select("customer_id", "customer_state").alias("cust"), "customer_id") \
    .join(df_order_reviews.select("order_id", "review_score", "review_comment_message"), "order_id", "left") \
    .select(
        "order_id", "seller_id", "cust.customer_state",
        "order_purchase_timestamp", "order_delivered_customer_date",
        "order_estimated_delivery_date", "order_status",
        "review_score", "review_comment_message"
    ) \
    .dropDuplicates(["order_id", "seller_id"])

df_gold_delivery_satisfaction.write.parquet('../data/gold/gold_delivery_satisfaction', mode='overwrite')

26/06/18 14:28:03 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


In [64]:
df_gold_payments = df_order_payments \
    .join(df_orders.select("order_id", "order_status", "customer_id", "order_purchase_timestamp"), "order_id", "left") \
    .join(df_customers.select("customer_id", "customer_state"), "customer_id", "left") \
    .select(
        "order_id", "payment_type", "payment_value",
        "payment_installments", "payment_sequential",
        "order_status", "order_purchase_timestamp", "customer_state"
    )
df_gold_payments.write.parquet('../data/gold/gold_payments', mode='overwrite')

In [65]:
df_cust = df_customers.select("customer_id", "customer_state", "customer_city", "customer_zip_code_prefix").alias("cust")
df_geo = df_geolocation.alias("geo")

df_gold_geo = df_items_base \
    .join(df_cust, "customer_id") \
    .join(df_geo, col("cust.customer_zip_code_prefix") == col("geo.geolocation_zip_code_prefix"), "left") \
    .select(
        "order_id", "price", "freight_value",
        "order_purchase_timestamp", "product_category_name_english",
        col("cust.customer_state"), col("cust.customer_city"),
        "seller_state", "seller_city",
        col("geo.geolocation_lat"), col("geo.geolocation_lng")
    )

df_gold_geo.write.parquet('../data/gold/gold_geo', mode='overwrite')

26/06/18 14:28:06 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


In [66]:
from pyspark.sql.functions import sum, avg, count, round, desc, trunc, when, datediff

# 1. CA total
df_gold_sales.agg(round(sum("price"), 2).alias("ca_total")).show()



+-------------+
|     ca_total|
+-------------+
|1.358764606E7|
+-------------+



In [67]:
# 2. CA mensuel
df_gold_sales \
    .withColumn("mois", trunc("order_purchase_timestamp", "MM")) \
    .groupBy("mois") \
    .agg(round(sum("price"), 2).alias("ca_mensuel")) \
    .orderBy("mois") \
    .show(24)

+----------+----------+
|      mois|ca_mensuel|
+----------+----------+
|2016-09-01|    267.36|
|2016-10-01|  48920.72|
|2016-12-01|      10.9|
|2017-01-01| 120124.22|
|2017-02-01| 246912.43|
|2017-03-01|  374255.3|
|2017-04-01| 359927.23|
|2017-05-01| 505781.16|
|2017-06-01| 432896.71|
|2017-07-01| 496431.58|
|2017-08-01| 573499.89|
|2017-09-01| 624222.69|
|2017-10-01| 664219.43|
|2017-11-01|1010271.37|
|2017-12-01| 743914.17|
|2018-01-01| 950030.36|
|2018-02-01| 844178.71|
|2018-03-01| 983153.54|
|2018-04-01| 996647.75|
|2018-05-01| 996517.68|
|2018-06-01| 865124.31|
|2018-07-01| 895507.22|
|2018-08-01| 854686.33|
|2018-09-01|     145.0|
+----------+----------+



In [68]:
# 3. Panier moyen
df_gold_sales.groupBy("order_id") \
    .agg(sum("price").alias("total")) \
    .agg(round(avg("total"), 2).alias("panier_moyen")) \
    .show()

+------------+
|panier_moyen|
+------------+
|      137.75|
+------------+



In [69]:
# 4. Top 10 catégories
df_gold_sales.groupBy("product_category_name_english") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

+-----------------------------+----------+
|product_category_name_english|        ca|
+-----------------------------+----------+
|                health_beauty|1258549.54|
|                watches_gifts|1204938.79|
|               bed_bath_table|1036574.08|
|               sports_leisure| 987863.98|
|         computers_accesso...| 911771.45|
|              furniture_decor| 728555.69|
|                   cool_stuff| 635202.86|
|                   housewares| 632050.16|
|                         auto| 592694.12|
|                 garden_tools| 484974.47|
+-----------------------------+----------+
only showing top 10 rows


In [70]:
# 5. Top 10 vendeurs
df_gold_sales.groupBy("seller_id") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

+--------------------+---------+
|           seller_id|       ca|
+--------------------+---------+
|4869f7a5dfa277a7d...|229472.63|
|53243585a1d6dc264...|222776.05|
|4a3ca9315b744ce9f...|200472.92|
|fa1c13f2614d7b5c4...|194042.03|
|7c67e1448b00f6e96...|187753.95|
|7e93a43ef30c4f03f...|176431.87|
|da8622b14eb17ae28...|159996.77|
|7a67c85e85bb2ce85...|141745.53|
|1025f0e2d44d7041d...|138968.55|
|955fee9216a65b617...| 135171.7|
+--------------------+---------+
only showing top 10 rows


In [71]:
# 6. Délai moyen de livraison
df_gold_delivery_satisfaction \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("delai", datediff("order_delivered_customer_date", "order_purchase_timestamp")) \
    .agg(round(avg("delai"), 1).alias("delai_moyen_jours")) \
    .show()

+-----------------+
|delai_moyen_jours|
+-----------------+
|             12.5|
+-----------------+



In [72]:
# 7. Taux de retard
df_gold_delivery_satisfaction \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("en_retard", when(
        col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1
    ).otherwise(0)) \
    .agg(round(avg("en_retard") * 100, 2).alias("taux_retard_pct")) \
    .show()

+---------------+
|taux_retard_pct|
+---------------+
|           8.02|
+---------------+



In [73]:
# 8. Note moyenne client
df_gold_delivery_satisfaction \
    .agg(round(avg("review_score"), 2).alias("note_moyenne")) \
    .show()

+------------+
|note_moyenne|
+------------+
|        4.09|
+------------+



In [74]:
# 9. Impact des retards sur la satisfaction
df_gold_delivery_satisfaction \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("en_retard", when(
        col("order_delivered_customer_date") > col("order_estimated_delivery_date"), "retard"
    ).otherwise("a_temps")) \
    .groupBy("en_retard") \
    .agg(round(avg("review_score"), 2).alias("note_moyenne")) \
    .show()

+---------+------------+
|en_retard|note_moyenne|
+---------+------------+
|   retard|        2.56|
|  a_temps|        4.27|
+---------+------------+



In [75]:
# 10. Répartition des paiements
df_gold_payments \
    .groupBy("payment_type") \
    .agg(
        count("order_id").alias("nb_commandes"),
        round(sum("payment_value"), 2).alias("montant_total")
    ) \
    .orderBy(desc("nb_commandes")) \
    .show()

+------------+------------+-------------+
|payment_type|nb_commandes|montant_total|
+------------+------------+-------------+
| credit_card|       76795|1.254208419E7|
|      boleto|       19784|   2869361.27|
|     voucher|        5775|    379436.87|
|  debit_card|        1529|    217989.79|
| not_defined|           3|          0.0|
+------------+------------+-------------+



In [76]:
# 11. CA par état brésilien
df_gold_sales \
    .groupBy("customer_state") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

+--------------+----------+
|customer_state|        ca|
+--------------+----------+
|            SP|5200592.28|
|            RJ|1823992.68|
|            MG|1584380.65|
|            RS| 750147.02|
|            PR| 683083.76|
|            SC| 520102.84|
|            BA| 511349.99|
|            DF| 302603.94|
|            GO| 294591.95|
|            ES| 275037.31|
+--------------+----------+
only showing top 10 rows


In [77]:
# 12. Taux de commandes par statut
df_gold_sales \
    .groupBy("order_status") \
    .agg(count("order_id").alias("nb_commandes")) \
    .orderBy(desc("nb_commandes")) \
    .show()

+------------+------------+
|order_status|nb_commandes|
+------------+------------+
|   delivered|      110145|
|     shipped|        1185|
|    canceled|         542|
|    invoiced|         359|
|  processing|         357|
| unavailable|           7|
|    approved|           3|
+------------+------------+



In [78]:
# 13. Note moyenne par catégorie
df_gold_delivery_satisfaction \
    .join(df_gold_sales.select("order_id", "product_category_name_english"), "order_id", "left") \
    .groupBy("product_category_name_english") \
    .agg(round(avg("review_score"), 2).alias("note_moyenne")) \
    .orderBy(desc("note_moyenne")) \
    .show(10)

+-----------------------------+------------+
|product_category_name_english|note_moyenne|
+-----------------------------+------------+
|            cds_dvds_musicals|        4.64|
|         fashion_childrens...|         4.5|
|         costruction_tools...|        4.44|
|         books_general_int...|        4.44|
|              books_technical|        4.36|
|               books_imported|        4.34|
|          luggage_accessories|        4.31|
|                      flowers|        4.31|
|                   food_drink|        4.31|
|         small_appliances_...|         4.3|
+-----------------------------+------------+
only showing top 10 rows


In [79]:
# 14. Délai moyen par état
df_gold_delivery_satisfaction \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("delai", datediff("order_delivered_customer_date", "order_purchase_timestamp")) \
    .groupBy("customer_state") \
    .agg(round(avg("delai"), 1).alias("delai_moyen_jours")) \
    .orderBy(desc("delai_moyen_jours")) \
    .show(10)

+--------------+-----------------+
|customer_state|delai_moyen_jours|
+--------------+-----------------+
|            RR|             29.3|
|            AP|             27.2|
|            AM|             26.4|
|            AL|             24.5|
|            PA|             23.7|
|            SE|             21.4|
|            MA|             21.4|
|            CE|             21.1|
|            AC|             21.0|
|            PB|             20.3|
+--------------+-----------------+
only showing top 10 rows


In [80]:
# 15. CA par vendeur et état
df_gold_sales \
    .groupBy("seller_id", "seller_state") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

+--------------------+------------+---------+
|           seller_id|seller_state|       ca|
+--------------------+------------+---------+
|4869f7a5dfa277a7d...|          SP|229472.63|
|53243585a1d6dc264...|          BA|222776.05|
|4a3ca9315b744ce9f...|          SP|200472.92|
|fa1c13f2614d7b5c4...|          SP|194042.03|
|7c67e1448b00f6e96...|          SP|187753.95|
|7e93a43ef30c4f03f...|          SP|176431.87|
|da8622b14eb17ae28...|          SP|159996.77|
|7a67c85e85bb2ce85...|          SP|141745.53|
|1025f0e2d44d7041d...|          SP|138968.55|
|955fee9216a65b617...|          SP| 135171.7|
+--------------------+------------+---------+
only showing top 10 rows
